# 🫀 Suy tim cấp tốc — Dự báo tử vong (ET4248 Lab, bản 5 buổi)

Pipeline đầy đủ: **EDA → Baseline (Chicco 2020) → SMOTE (Ishaq 2021) → XAI (SHAP) → Báo cáo**.

Dataset: [Heart Failure Clinical Data](https://www.kaggle.com/datasets/andrewmvd/heart-failure-clinical-data) — 299 bệnh nhân, 13 đặc trưng, nhãn `DEATH_EVENT`.

**Mốc cần tiệm cận**
- Chicco 2020 (Random Forest, *không* dùng `time`): Accuracy ≈ **0.740**, MCC ≈ **0.384**
- Ishaq 2021 (SMOTE + cây): Accuracy ≈ **0.926**

> ⚠️ **Hai lỗi bị trừ điểm nặng trong rubric:**
> 1. **Data leakage** — `StandardScaler` / `SMOTE` phải `fit` **chỉ trên tập Train**.
> 2. **SMOTE trên tập Test** — tuyệt đối không. Test phải giữ nguyên bản.

Chạy lần lượt từ trên xuống. Mỗi buổi = một sản phẩm bàn giao (`.ipynb`).

---
## Buổi 0 — Cài đặt môi trường & nạp dữ liệu

In [ ]:
# Colab đã có sẵn pandas/sklearn. Chỉ cần thêm 2 thư viện này:
!pip install -q imbalanced-learn shap kagglehub

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (8, 5)
print("Setup OK")

### Nạp dữ liệu — chọn 1 trong 2 cách
- **Cách A (tự động):** `kagglehub` tải trực tiếp từ Kaggle (có thể hỏi đăng nhập Kaggle lần đầu).
- **Cách B (thủ công):** tải file CSV từ Kaggle về máy → chạy ô upload bên dưới.

In [ ]:
df = None

# --- Cách A: kagglehub ---
try:
    import kagglehub
    path = kagglehub.dataset_download("andrewmvd/heart-failure-clinical-data")
    for f in os.listdir(path):
        if f.endswith(".csv"):
            df = pd.read_csv(os.path.join(path, f))
            print("✅ Nạp qua kagglehub:", f)
            break
except Exception as e:
    print("kagglehub chưa dùng được:", e)

# --- Cách B: upload thủ công (chỉ chạy nếu Cách A thất bại) ---
if df is None:
    from google.colab import files
    up = files.upload()                      # chọn file .csv tải từ Kaggle
    df = pd.read_csv(list(up.keys())[0])
    print("✅ Nạp qua upload thủ công")

print("Kích thước:", df.shape)   # kỳ vọng (299, 13)
df.head()

---
## Buổi 1 — Khám phá & Chuẩn bị dữ liệu (EDA & Preprocessing)
**Sản phẩm:** `1_eda.ipynb`

### Nhiệm vụ 1 — EDA

In [ ]:
# Kiểu dữ liệu & giá trị khuyết
df.info()
print("\nSố giá trị khuyết mỗi cột:")
print(df.isnull().sum())

In [ ]:
# Tỷ lệ nhãn -> dữ liệu có mất cân bằng không?
counts = df["DEATH_EVENT"].value_counts()
ratio  = df["DEATH_EVENT"].value_counts(normalize=True).round(4)
print(pd.concat([counts, ratio], axis=1, keys=["count", "ratio"]))

ax = counts.plot(kind="bar", color=["#4C72B0", "#C44E52"])
ax.set_xticklabels(["0 = Survived", "1 = Dead"], rotation=0)
ax.set_title("Phân bố nhãn DEATH_EVENT (mất cân bằng ~68/32)")
plt.show()

In [ ]:
# Phân bố các biến số liên tục
num_cols = ["age", "creatinine_phosphokinase", "ejection_fraction",
            "platelets", "serum_creatinine", "serum_sodium", "time"]
df[num_cols].hist(bins=30, figsize=(14, 8))
plt.tight_layout(); plt.show()

In [ ]:
# Ma trận tương quan (Correlation Heatmap)
plt.figure(figsize=(11, 9))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, square=True)
plt.title("Correlation Heatmap")
plt.show()

# Tương quan của từng biến với nhãn, xếp theo |giá trị|
corr_target = df.corr(numeric_only=True)["DEATH_EVENT"].drop("DEATH_EVENT")
print(corr_target.reindex(corr_target.abs().sort_values(ascending=False).index).round(3))

**Nhận xét gợi ý:** `time`, `serum_creatinine`, `ejection_fraction`, `age` tương quan mạnh nhất với `DEATH_EVENT`.
`serum_creatinine` & `ejection_fraction` chính là 2 biến mà bài báo Chicco 2020 nhấn mạnh.

### Nhiệm vụ 2 — Train/Test Split (stratified 80/20)

> **Về cột `time` (thời gian theo dõi):** đây là biến rất mạnh nhưng gây tranh cãi — nó phản ánh *kết cục sau khi đã theo dõi*, nên đưa vào mô hình "dự báo lúc nhập viện" là một dạng rò rỉ thông tin.
> Để **tái lập đúng con số 74% của Chicco**, ta **bỏ `time`** ở Buổi 2. Ta vẫn giữ 1 phiên bản *có* `time` để so sánh và bàn luận.

In [ ]:
from sklearn.model_selection import train_test_split

y = df["DEATH_EVENT"]
X_full = df.drop(columns=["DEATH_EVENT"])         # có time
X_clin = df.drop(columns=["DEATH_EVENT", "time"]) # bỏ time (theo Chicco)

# Chia CÙNG chỉ số cho cả hai phiên bản để so sánh công bằng
idx_tr, idx_te = train_test_split(
    df.index, test_size=0.20, stratify=y, random_state=RANDOM_STATE)

X_train,  X_test  = X_clin.loc[idx_tr], X_clin.loc[idx_te]   # dùng chính cho Buổi 2-4
Xf_train, Xf_test = X_full.loc[idx_tr], X_full.loc[idx_te]   # phiên bản có time
y_train,  y_test  = y.loc[idx_tr],  y.loc[idx_te]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Tỷ lệ nhãn Train:", y_train.value_counts(normalize=True).round(3).to_dict())
print("Tỷ lệ nhãn Test :", y_test.value_counts(normalize=True).round(3).to_dict())

### Nhiệm vụ 3 — Chuẩn hóa (StandardScaler)

**Câu hỏi tư duy — tại sao chỉ `fit` trên Train?**
`fit` học `mean`/`std` từ dữ liệu. Nếu học cả trên Test, thông tin của Test "rò rỉ" vào bước tiền xử lý → mô hình gián tiếp *nhìn thấy* Test khi huấn luyện → điểm đánh giá bị thổi phồng, không phản ánh hiệu năng thực tế trên bệnh nhân mới. Test phải đóng vai "dữ liệu chưa từng thấy".

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)      # fit CHỈ trên Train
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_s  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)
print("Đã chuẩn hóa. Mean(Train)≈0:", X_train_s.mean().round(2).to_dict())

---
## Buổi 2 — Tái lập Y văn gốc (Baseline)
**Sản phẩm:** `2_baseline.ipynb` + bảng so sánh

Mục tiêu: Logistic Regression + Random Forest, đo `Accuracy`, `MCC`, `F1`, đối chiếu Chicco (0.740 / 0.384).

In [ ]:
from sklearn.metrics import accuracy_score, matthews_corrcoef, f1_score, classification_report

def evaluate(name, model, X_tr, y_tr, X_te, y_te, show_report=False):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    row = {"model": name,
           "accuracy": round(accuracy_score(y_te, pred), 3),
           "MCC":      round(matthews_corrcoef(y_te, pred), 3),
           "F1":       round(f1_score(y_te, pred), 3)}
    if show_report:
        print(classification_report(y_te, pred, target_names=["Survived", "Dead"]))
    return row, model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

results = []

# Logistic Regression cần dữ liệu đã chuẩn hóa
r, _ = evaluate("LogReg (no time)",
                LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                X_train_s, y_train, X_test_s, y_test)
results.append(r)

# Random Forest không cần chuẩn hóa (dùng X gốc)
r, rf_model = evaluate("RandomForest (no time)",
                RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
                X_train, y_train, X_test, y_test, show_report=True)
results.append(r)

pd.DataFrame(results)

**Lưu ý về việc "khớp số":** Chicco lấy **trung bình 100 lần chạy** với các cách chia khác nhau, nên 1 lần chia 80/20 sẽ dao động. Ô dưới lặp lại nhiều lần chia để lấy trung bình MCC — sát tinh thần bài báo hơn.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

def repeated_mean(model_fn, X, y, scale=False, n=100):
    sss = StratifiedShuffleSplit(n_splits=n, test_size=0.2, random_state=RANDOM_STATE)
    accs, mccs, f1s = [], [], []
    for tr, te in sss.split(X, y):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        if scale:
            sc = StandardScaler().fit(Xtr)
            Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
        m = model_fn().fit(Xtr, ytr); p = m.predict(Xte)
        accs.append(accuracy_score(yte, p)); mccs.append(matthews_corrcoef(yte, p)); f1s.append(f1_score(yte, p))
    return {"accuracy": round(np.mean(accs),3), "MCC": round(np.mean(mccs),3), "F1": round(np.mean(f1s),3)}

print("RandomForest (no time), TB 100 lần:",
      repeated_mean(lambda: RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
                    X_clin, y))

### (Bàn luận) Điều gì xảy ra nếu THÊM `time`?

In [ ]:
r_ft, _ = evaluate("RandomForest (+time)",
                   RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
                   Xf_train, y_train, Xf_test, y_test)
print(r_ft)
print("→ Accuracy tăng vọt vì 'time' rò rỉ kết cục. Đây là bài học về leakage, không phải cải tiến hợp lệ.")

---
## Buổi 3 — Tối ưu & Cải tiến với SMOTE
**Sản phẩm:** `3_improvement.ipynb` + bảng "Có SMOTE vs Không SMOTE"

Mục tiêu: xử lý mất cân bằng bằng **SMOTE (chỉ trên Train)**, thử Extra Trees / Gradient Boosting, tiệm cận Ishaq (0.926).

> 🚩 **Red flag:** `SMOTE.fit_resample` chỉ gọi trên `X_train, y_train`. `X_test` giữ nguyên.

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)   # CHỈ Train
print("Trước SMOTE:", y_train.value_counts().to_dict())
print("Sau  SMOTE:", pd.Series(y_train_res).value_counts().to_dict())

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier

def eval_pred(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr); p = model.predict(Xte)
    return {"model": name,
            "accuracy": round(accuracy_score(yte, p),3),
            "MCC": round(matthews_corrcoef(yte, p),3),
            "F1": round(f1_score(yte, p),3)}, model

cmp = []
# Không SMOTE
cmp.append(eval_pred("ExtraTrees (no SMOTE)",
        ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
        X_train, y_train, X_test, y_test)[0])
# Có SMOTE
et_row, et_smote = eval_pred("ExtraTrees + SMOTE",
        ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE),
        X_train_res, y_train_res, X_test, y_test)
cmp.append(et_row)
cmp.append(eval_pred("GradBoost + SMOTE",
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        X_train_res, y_train_res, X_test, y_test)[0])
cmp.append(eval_pred("RandomForest + SMOTE",
        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        X_train_res, y_train_res, X_test, y_test)[0])

pd.DataFrame(cmp)

**Ablation nhỏ với `n_estimators`** (yêu cầu Nhiệm vụ 3):

In [ ]:
for n in [50, 100, 200, 300, 500]:
    m = ExtraTreesClassifier(n_estimators=n, random_state=RANDOM_STATE).fit(X_train_res, y_train_res)
    p = m.predict(X_test)
    print(f"n_estimators={n:>3}  acc={accuracy_score(y_test,p):.3f}  MCC={matthews_corrcoef(y_test,p):.3f}")

**Cách đúng để tinh chỉnh mà KHÔNG leak** — dùng `imblearn.Pipeline` để SMOTE chạy *bên trong* mỗi fold của cross-validation:

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipe = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("clf",   ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE)),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = cross_val_score(pipe, X_clin, y, cv=cv, scoring="f1")
print("F1 (5-fold, SMOTE trong pipeline):", scores.round(3), " TB:", round(scores.mean(),3))

> **Trung thực khoa học (rubric #5):** đạt đúng 92.6% không phải lúc nào cũng dễ chỉ với SMOTE trên tập test cố định. Con số cao nhất trong y văn thường đi kèm chọn đặc trưng và/hoặc dùng `time`. Hãy báo cáo con số *thật* bạn đạt được và giải thích khoảng cách.

---
## Buổi 4 — Giải thích mô hình (XAI) với SHAP
**Sản phẩm:** `4_xai.ipynb` + `shap_summary.png` + bình luận đối chiếu

In [ ]:
import shap
print("SHAP version:", shap.__version__)

# Dùng mô hình cây tốt nhất từ Buổi 3
explainer = shap.TreeExplainer(et_smote)
shap_values = explainer.shap_values(X_test)

# Chuẩn hóa shape cho lớp dương (Dead=1) trên các phiên bản SHAP khác nhau
def positive_class_sv(sv):
    if isinstance(sv, list):          # [class0, class1]
        return sv[1]
    if getattr(sv, "ndim", 2) == 3:   # (n, features, classes)
        return sv[:, :, 1]
    return sv
sv_pos = positive_class_sv(shap_values)
print("SHAP values shape:", np.array(sv_pos).shape)

In [ ]:
shap.summary_plot(sv_pos, X_test, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu shap_summary.png")

In [ ]:
# Xếp hạng tầm quan trọng = trung bình |SHAP|
importance = (pd.Series(np.abs(sv_pos).mean(axis=0), index=X_test.columns)
                .sort_values(ascending=False))
print(importance.round(4))

top2 = importance.index[:2].tolist()
print("\nTop-2 đặc trưng:", top2)
for f in ["serum_creatinine", "ejection_fraction"]:
    print(f"  {f}: hạng {list(importance.index).index(f)+1}/{len(importance)}")

### Nhiệm vụ 3 — Đối chiếu lâm sàng (mẫu bình luận, sửa theo kết quả của bạn)

> Bài báo Chicco 2020 kết luận **serum creatinine** và **ejection fraction** là 2 yếu tố quyết định nhất.
> Trên biểu đồ SHAP của mô hình Extra Trees + SMOTE, hai biến này nằm ở nhóm quan trọng nhất → mô hình học đúng sinh lý bệnh, không "học vẹt".
>
> **Nếu KHÔNG khớp:** SMOTE tạo mẫu tổng hợp bằng nội suy giữa các điểm thiểu số, có thể làm *biến dạng phân bố gốc* của một số đặc trưng và đẩy thứ hạng lệch đi. Có thể kiểm chứng bằng cách chạy lại SHAP trên mô hình **không SMOTE** (`ExtraTrees (no SMOTE)`) và so sánh thứ hạng.

In [ ]:
# (Tùy chọn) SHAP trên mô hình KHÔNG SMOTE để so sánh biến dạng
et_nosmote = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE).fit(X_train, y_train)
sv2 = positive_class_sv(shap.TreeExplainer(et_nosmote).shap_values(X_test))
imp2 = pd.Series(np.abs(sv2).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
pd.DataFrame({"rank_SMOTE": importance.rank(ascending=False).astype(int),
              "rank_noSMOTE": imp2.rank(ascending=False).astype(int)}).sort_values("rank_SMOTE")

---
## Buổi 5 — Báo cáo & Nghiệm thu

Xuất báo cáo 5–10 trang (Slide/PDF) gồm 6 phần theo rubric. Checklist:

1. **Đặt vấn đề** — bài toán dự báo tử vong suy tim, ý nghĩa lâm sàng (hỗ trợ bác sĩ ưu tiên bệnh nhân nguy cơ cao).
2. **Y văn** — Chicco 2020 (RF, 0.740 / MCC 0.384; nhấn mạnh serum creatinine + ejection fraction) và Ishaq 2021 (SMOTE, ~0.926).
3. **Phương pháp** — stratified 80/20; `StandardScaler` fit-Train-only; SMOTE chỉ trên Train.
4. **Kết quả** — bảng ghép LogReg / RandomForest / ExtraTrees+SMOTE, đối chiếu 2 mốc y văn.
5. **XAI** — chèn `shap_summary.png`, diễn giải vai trò serum creatinine & ejection fraction.
6. **Hạn chế** — cỡ mẫu nhỏ (299), dữ liệu 2015, một trung tâm (Pakistan), thiếu validation ngoài; rủi ro leakage từ `time`.

Ô dưới gộp toàn bộ số liệu để dán thẳng vào báo cáo.

In [ ]:
summary = pd.DataFrame(results + cmp)
print("=== BẢNG KẾT QUẢ TỔNG HỢP ===")
print(summary.to_string(index=False))
summary.to_csv("results_summary.csv", index=False)
print("\nĐã lưu results_summary.csv (tải về từ panel Files bên trái).")

### Tải sản phẩm về máy

In [ ]:
from google.colab import files
for f in ["shap_summary.png", "results_summary.csv"]:
    try: files.download(f)
    except Exception as e: print("Bỏ qua", f, e)

---
**Tài liệu tham khảo**
- Chicco & Jurman (2020), *BMC Med Inform Decis Mak* 20:16.
- Ishaq et al. (2021), *IEEE Access* — SMOTE + data mining cho survival suy tim.
- Đề bài: `fossbk-spec.github.io/hmyt-book/du_an/suy_tim_risk_dxai_lab`